# NSE ETF Statistics - By Indranil

## Pre-requisite

### Setup the environment variables and Databricks Secrets:


1. Open databricks notebooks and open terminal from bottom left corner of databricks notebooks
    ```cmd
    <!-- databricks secrets create-scope <ScopeName> --initial-manage-principal users -->
    databricks secrets create-scope Databricks_Scope --initial-manage-principal users

    <!-- databricks secrets put-secret <scopeName> <SecretName> -->
    databricks secrets put-secret Databricks_Scope GEMINI_API_KEY

    databricks secrets list-scopes
    databricks secrets list-secrets Databricks_Scope
    databricks secrets delete-scope --scope market_stat_scope
    ```
    

## Step 1: Prepare Market Data From NSE

In [0]:
from utility.trading.nse_api import NSE_API


In [0]:
from utility.trading.nse_api import load_etf_data
from utility.trading.nse_api import get_nse_etf_data

# final_etf_df = get_nse_etf_data()
final_etf_df_new_filtered = load_etf_data()

final_etf_df_new_filtered.shape

## Step 2: Core Analysis By LLM Model(Gemini)

In [0]:
GEMINI_API_KEY = dbutils.secrets.get(scope="Databricks_Scope", key="GEMINI_API_KEY")
GEMINI_MODEL = "gemini-2.5-flash"  #  "gemini-2.5-flash-preview-09-2025"
API_URL = f"https://generativelanguage.googleapis.com/v1beta/models/{GEMINI_MODEL}:generateContent?key={GEMINI_API_KEY}"

In [0]:
import json
import time
import requests


def get_gemini_etf_analysis(etf_df, user_goal, rag_context=""):
    """
    Calls the Gemini API with structured data, RAG context, and a robust prompt.

    Args:
        etf_df (pd.DataFrame): The structured ETF data.
        user_goal (str): The specific investment objective (e.g., "long-term growth").
        rag_context (str): Contextual information retrieved from a RAG system (e.g., tax laws).
    """
    # Convert DataFrame to a structured, LLM-readable format (Markdown table or JSON string)
    etf_data_markdown = "### ETF Data for Analysis\n\n" + etf_df.to_markdown(index=False)
    
    # --- The Optimized Prompt (for detail, comparison, and structure) ---

    # system_instruction = (
    #     "You are a Senior Certified Financial Analyst (CFA) specializing in ETF portfolio strategy with a dedicated focus on the Indian (NSE) market. "
    #     "Your analysis must be grounded in information from official Indian financial sources (e.g., NSE India, AMFI, major Indian brokerage sites). "
    #     "Your primary task in Step 3 is to use the Google Search tool to find the latest Expense Ratio for each of the provided ETF tickers, assuming their nearest NSE equivalents. "
    #     "You **MUST** limit your search scope by using the keyword 'Expense Ratio NSE' for each ETF and prioritize information from high-authority sources in India. "
    #     "Your response must be analytical, detailed, and structured in a step-by-step format. "
    #     "Adhere strictly to the requested output structure. **DO NOT** output any internal thought processes, planning steps, or code execution blocks."
    # )

    system_instruction = (
        "You are a Senior Certified Financial Analyst (CFA) specializing in ETF portfolio strategy with a dedicated focus on the Indian (NSE) market. "
        "Your analysis must be grounded in information from official Indian financial sources (e.g., NSE India, AMFI, major Indian brokerage sites). "
        "Your primary task is to analyze the PROVIDED ETF data against the investment goal. "
        "Your response must be analytical, detailed, and structured in a step-by-step format. "
        "Adhere strictly to the requested output structure. **DO NOT** output any internal thought processes, planning steps, or code execution blocks."
    )
    
    user_query = f"""
    Based on the following ETF data and my investment goal:
    
    1.  **ETF Data:**
        {etf_data_markdown}

    2.  **Investment Goal:** 
        {user_goal}

    3.  **Contextual RAG Data (Finance/Tax/Policy):**
        {rag_context if rag_context else "No specific external context provided."}
        
    **REQUIRED OUTPUT FORMAT (Step-by-Step):**
    
    --- START ANALYSIS ---
    
    ## Step 1: Summary of Investment Goal and Data Context
    (A brief, 2-3 sentence overview of the goal and the dataset analyzed.)
    
    ## Step 2: Comparative Performance Analysis
    (Compare key metrics: YTD Return(yearly_percentage_change), MTD Return(monthly_percentage_change), iNAV(Indicative Net Asset Value), Total Market Cap, Trade Volume  and Expense Ratio for all provided ETFs. Discuss risk/reward trade-offs.)

    ## Step 3: Deep Dive and Recommendation
    (Recommend the single best ETF that aligns with the '{user_goal}' goal. **CRITICAL: Base your analysis solely on the PROVIDED ETF Data. The recommendation MUST favor the ETF with the lowest expense ratio, highest AUM, and superior liquidity/trading volume (inferred from AUM/volume data).** You must present the key decision metrics clearly.)
    
    
    ## Step 4: Tax and Financial Implications (Integrating RAG)
    (Discuss the tax implications of the recommended ETF (e.g., dividend taxation, capital gains) and any relevant policies from the 'Contextual RAG Data' section.)
    
    ## Step 5: Visualization Guidance
    (Describe, in detail, the specific type of chart/diagram (e.g., a clustered bar chart or a waterfall chart) that a Databricks developer should generate to visually summarize the comparative performance and sector exposure data. Do not generate the code, just the description.)

    --- END ANALYSIS ---
    """
    
    # --- API Payload with Grounding and System Instruction ---
    payload = {
        "contents": [{
            "parts": [{"text": user_query}]
        }],
        "tools": [{
            "google_search": {} # Enables Google Search for grounding the recommendation
        }],
        "systemInstruction": {
            "parts": [{"text": system_instruction}]
        }
    }

    print(f"\nCalling Gemini API ({GEMINI_MODEL}) for ETF analysis...")
    
    # Implement exponential backoff for robustness
    max_retries = 3
    for attempt in range(max_retries):
        try:
            response = requests.post(API_URL, headers={'Content-Type': 'application/json'}, data=json.dumps(payload))
            response.raise_for_status() # Raises an exception for HTTP error codes
            result = response.json()
            
            # Extract generated text
            generated_text = result['candidates'][0]['content']['parts'][0]['text']
            
            # Extract and format Grounding Sources
            sources = []
            grounding_metadata = result['candidates'][0].get('groundingMetadata')
            if grounding_metadata and grounding_metadata.get('groundingAttributions'):
                sources = [
                    f"Source: {attr['web']['title']} - {attr['web']['uri']}" 
                    for attr in grounding_metadata['groundingAttributions']
                ]

            print("\n" + "="*80)
            print("                ✅ GEMINI ETF ANALYSIS SUCCESSFUL ✅")
            print("="*80 + "\n")
            print(result)
            print("="*80 + "\n")
            
            if sources:
                print("\n\n--- 🌐 Real-Time Grounding Sources (Google Search) ---")
                for src in sources:
                    print(src)
            print("\n" + "="*80)
            return generated_text

        except requests.exceptions.RequestException as e:
            if attempt < max_retries - 1:
                wait_time = 2 ** attempt
                print(f"Attempt {attempt + 1} failed (Error: {e}). Retrying in {wait_time} seconds...")
                time.sleep(wait_time)
            else:
                print(f"Final attempt failed. Could not connect to Gemini API. Error: {e}")
                return None
        except Exception as e:
            print(f"An unexpected error occurred during API processing: {e}")
            return None


In [0]:
# from utility.trading.nse_api import load_etf_data

# Load Data
# etf_data = final_etf_df
etf_data = final_etf_df_new_filtered

# Define the User's Goal
# investment_goal =  "Aggressive long-term growth and capital appreciation for a Roth IRA retirement account."
investment_goal =  "Aggressive long-term growth and capital appreciation for retirement planning in India."

# This information would typically be fetched from a Vector Database query.
rag_context_data = """
    [RAG Context Document 1: Indian ETF Capital Gains Taxation (FY 2024-25)]
    1. Equity ETFs (Invests >= 65% in Indian Equity):
        - STCG (Short-Term Capital Gains, held ≤ 12 months): Taxed at a flat rate of 20% (u/s 111A).
        - LTCG (Long-Term Capital Gains, held > 12 months): Gains up to ₹1,25,000 per financial year are exempt. Gains exceeding this limit are taxed at 10% (without indexation benefit).
    2. Debt/Non-Equity ETFs (Gold, International, etc.):
        - Investments made on or after April 1, 2023: All capital gains (STCG/LTCG, regardless of holding period) are treated as Short-Term Capital Gains (STCG) and are taxed at the investor's marginal Income Tax Slab Rate. No indexation benefit is available.
    3. Dividend Income: All dividend income from any ETF is added to the investor's total income and taxed at the applicable Income Tax Slab Rate.

    [RAG Context Document 2: Indian Financial Planning Principles]
    - Compounding and SIP (Systematic Investment Plan) are the preferred methods for long-term wealth creation in India.
    - **Tax Loss Harvesting:** Short-term capital losses can be set off against both short-term and long-term capital gains. Long-term capital losses can only be set off against long-term gains. Losses can be carried forward for 8 years.
    - **Asset Allocation:** For long-term goals (10+ years), a higher allocation to equity ETFs is typically recommended to beat inflation and generate superior real returns.

    [RAG Context Document 3: Financial Glossary - CAGR]
    Compound Annual Growth Rate (CAGR) is the average annual return of an investment over a specified period longer than one year. A higher CAGR indicates superior historical performance.
"""

generated_text_output = get_gemini_etf_analysis(
                            etf_df=etf_data,
                            user_goal=investment_goal,
                            rag_context=rag_context_data
                        )


# In a real Databricks notebook, you would now use the generated Step 5 guidance
# (e.g., 'Clustered Bar Chart') to programmatically generate the chart 
# using Matplotlib, Plotly, or Databricks native display functions.
# Example: display(etf_data.plot.bar(x='ticker', y='ytd_return_percent'))

In [0]:
# print("\n" + "="*100)
# print("                         ✅ GEMINI ETF ANALYSIS SUCCESSFUL ✅.    ")
# print("="*100 + "\n")
print(generated_text_output)

## Sample

In [0]:
from utility.trading.nse_api import get_nse_etf_data_monthly_ohlc

out = get_nse_etf_data_ohlc(symbol='MON100')
out

 Step 3: Deep Dive and Recommendation
 ---
    (Recommend the single best ETF that aligns with the '{user_goal}' goal. **CRITICAL: Use the Google Search tool with the explicit search terms 'iNAV NSE [ETF Ticker]', 'Expense Ratio NSE [ETF Ticker]', AND 'AUM NSE [ETF Ticker]' to find the *latest iNAV, Expense Ratio, AND AUM on the NSE*. Compare the iNAV against the last traded value to determine the trading premium/discount. The recommendation MUST favor the ETF with the low expense ratio, good AUM, and the highest liquidity/trading volume in its sector.** You must present the search results and analysis clearly. Use Google Search for real-time market news to ground your recommendation.)

1. snowflake pandas modin
2. databricks Secret manager
3. stich.ai, Pomelli.ai, Opal.hoogle, Notebook LLM